In [1]:
# How do I increase the cell width of the Jupyter version 7?
# https://stackoverflow.com/a/78487654/2879865

# # EXPANDING Jupyter Notebook Cells to the width of the screen

# # Till versions below 7
# from IPython.display import display, HTML
# display(HTML("<style>.container { width:101.25% !important; }</style>"))

# # From above version 7 (this has some DeprecationWarning)
# from IPython.core.display import display, HTML
# display(HTML("<style>.jp-Cell { margin-left: -17% !important; margin-right: -12.75% !important; }</style>"))

# From version 7.14 and above
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -4.25% !important; margin-right: -0.75% !important; }</style>"))

%autosave 10

Autosaving every 10 seconds


In [2]:
from time import time
import os
import platform

print("Operating System:", platform.system())
print("Architecture:", platform.architecture()[0])
print("Python Version:", platform.python_version())
print("CPU Count:", os.cpu_count())
# The os.cpu_count() function in Python returns the number of logical CPUs available on the system. This includes all cores as well as any logical processors created by hyper-threading.
# So, if your CPU has hyper-threading, os.cpu_count() might return a number greater than the physical core count. To get the actual number of physical cores, you might need to use a different method, such as accessing system-specific information through libraries like psutil or using platform-specific tools.

print(os.getcwd()) # Check if local compute infrastructure is being used or cloud
# heuristic_start_time = time()
# print(heuristic_start_time)

Operating System: Windows
Architecture: 64bit
Python Version: 3.12.7
CPU Count: 4
C:\Users\Santanu Banerjee\SuperPermutation


In [3]:
import pandas as pd
# import numpy as np
import random
import math
import matplotlib.pyplot as plt
# import matplotlib.font_manager as font_manager
from matplotlib.ticker import MaxNLocator, FixedLocator, FixedFormatter, FormatStrFormatter
import ast
import sys
import itertools
import gurobipy as gupy
from gurobipy import *
import csv
import openpyxl


print(sys.version) # Correct usage
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']

3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]


In [4]:
# current_N = 4

# # Better bounds are available according to https://oeis.org/A180632
# upper_Bound = 33 # https://oeis.org/A007489
# known_Lower_Bound = 33 # https://oeis.org/A376269

# starting_Solution = "123412314231243121342132413214321"

In [5]:
current_N = 5

# Better bounds are available according to https://oeis.org/A180632
upper_Bound = 153 # https://oeis.org/A007489
known_Lower_Bound = 152 # https://oeis.org/A376269

starting_Solution = "123451234152341253412354123145231425314235142315423124531243512431524312543121345213425134215342135421324513241532413524132541321453214352143251432154321"

starting_Solution_2 = "123451234152341253412354123145231425314235142315423124531243512431524312543121354213524135214352134521325413251432513425132451321543215342153241532145321"

In [6]:
M = 10E7 # Big M

if len(starting_Solution) < upper_Bound:
    raise ValueError("starting_Solution is too short")
else:
    upper_Bound = len(starting_Solution)
    print(upper_Bound)

153


In [7]:
elements = range(1, current_N+1)
permS = list(itertools.permutations(elements))

if len(permS) != math.factorial(current_N):
    print("Error")
    sys.exit(1)
# for p in perms:
#     print(p)

In [8]:
def callback_fun(mdl, where):
    # Check if a new integer feasible solution has been found
    if where == GRB.Callback.MIPSOL:
        # Create or append to the CSV file
        csv_file = os.path.join(Model_and_Solution_Folder, "Solution_Times.csv")
        
        # Write header if file does not exist
        if not os.path.exists(csv_file):
            with open(csv_file, "w", newline='') as fq:
                writer = csv.writer(fq)
                writer.writerow([
                    "Callback Solution Count (Gurobi)",
                    "Current Best Obj. Val.",
                    "New Obj. Val.",
                    "Run Time (minutes)",
                    "Solver Work Units",
                    "Best Bound",
                    "Current Phase: 0 (in the NoRel heuristic), 1 (in the standard MIP search), or 2 (performing MIP solution improvement)",
                    "Explored Node Count"
                ])
        
        # Write data row
        with open(csv_file, "a", newline='') as fq:
            writer = csv.writer(fq)
            writer.writerow([
                mdl.cbGet(GRB.Callback.MIPSOL_SOLCNT),
                mdl.cbGet(GRB.Callback.MIPSOL_OBJBST),
                mdl.cbGet(GRB.Callback.MIPSOL_OBJ),
                mdl.cbGet(GRB.Callback.RUNTIME)/60,
                mdl.cbGet(GRB.Callback.WORK),
                mdl.cbGet(GRB.Callback.MIPSOL_OBJBND),
                mdl.cbGet(GRB.Callback.MIPSOL_PHASE),
                mdl.cbGet(GRB.Callback.MIPSOL_NODCNT)
            ])

In [9]:
def model_development():
    prob = gupy.Model(f"Super-Permutation_{current_N}")
    
    x={}
    f={}
    y={}
    h={}
    g={}
    z={}

    
    # Variable Declarations:
    for j in range(1, upper_Bound+1):
        x[j] = prob.addVar(lb=0, ub=current_N, vtype=gupy.GRB.INTEGER, name=f'x[{j}]') # Eq. 4a
        x[j].Start = int(starting_Solution[j - 1]) # Assigning start value directly — safe and clean
        f[j] = prob.addVar(vtype=gupy.GRB.BINARY, name=f'f[{j}]') # Eq. 4b
        
    S_comparison_with_X_UpperBound = upper_Bound-current_N+1
    for j in range(1, S_comparison_with_X_UpperBound+1):
        for s in permS:
            for i in elements:
                y[s,j,i] = prob.addVar(lb=-current_N, ub=current_N, vtype=gupy.GRB.CONTINUOUS, name=f'y[{s},{j},{i}]') # Eq. 4c
                g[s,j,i] = prob.addVar(vtype=gupy.GRB.BINARY, name=f'g[{s},{j},{i}]') # Eq. 4d
                h[s,j,i] = prob.addVar(vtype=gupy.GRB.BINARY, name=f'h[{s},{j},{i}]') # Eq. 4d
            z[s,j] = prob.addVar(vtype=gupy.GRB.BINARY, name=f'z[{s},{j}]') # Eq. 4e
        
    
    
    # Constraint Declarations
    for s in permS:
        for i in elements:
            for j in range(1, S_comparison_with_X_UpperBound+1):
                prob.addConstr(x[j+i-1] - s[i-1] == y[s,j,i], name=f"Eq.1: s={s}, j={j}, i={i}")  # Equation 1
                
                prob.addConstr(y[s,j,i] <= M*h[s,j,i], name=f"Eq.2a: s={s}, j={j}, i={i}")  # Equation 2a
                prob.addConstr(y[s,j,i] >= M*(h[s,j,i]-1), name=f"Eq.2b: s={s}, j={j}, i={i}")  # Equation 2b
                prob.addConstr(g[s,j,i] <= y[s,j,i] + M*(1-h[s,j,i]), name=f"Eq.2c: s={s}, j={j}, i={i}")  # Equation 2c
                prob.addConstr(M*g[s,j,i] >= y[s,j,i] - M*(1-h[s,j,i]), name=f"Eq.2d: s={s}, j={j}, i={i}")  # Equation 2d
                prob.addConstr(g[s,j,i] <= M*h[s,j,i] - y[s,j,i], name=f"Eq.2e: s={s}, j={j}, i={i}")  # Equation 2e
                prob.addConstr(M*g[s,j,i] >= -M*h[s,j,i] - y[s,j,i], name=f"Eq.2f: s={s}, j={j}, i={i}")  # Equation 2f
        
        for j in range(1, S_comparison_with_X_UpperBound+1):
            prob.addConstr(gupy.quicksum(g[s,j,i] for i in elements) <= M*(1-z[s,j]), name=f"Eq.3a: s={s}, j={j}")  # Equation 3a
            prob.addConstr(gupy.quicksum(g[s,j,i] for i in elements) >= 1-z[s,j], name=f"Eq.3b: s={s}, j={j}")  # Equation 3b
        prob.addConstr(gupy.quicksum(z[s,j] for j in range(1, S_comparison_with_X_UpperBound+1)) >= 1, name=f"Eq.3c: s={s}")  # Equation 3c
    


    
    # Optimization Objective
    for j in range(1, upper_Bound+1):
        prob.addConstr(f[j] <= x[j], name=f"Eq.5a: j={j}")  # Equation 5a
        prob.addConstr(x[j] <= current_N*f[j], name=f"Eq.5b: j={j}")  # Equation 5b
    prob.setObjective(gupy.quicksum(f[j] for j in range(1, upper_Bound+1)), sense=gupy.GRB.MINIMIZE) # Equation 5c
    prob.addConstr(gupy.quicksum(f[j] for j in range(1, upper_Bound+1)) >= known_Lower_Bound, name=f"Objective KNOWN Lower Bound") # Lower Bound of Objective Function
 
    
        
    if not os.path.exists(Model_and_Solution_Folder+"/Intermediate_Solutions"):
        os.mkdir(Model_and_Solution_Folder+"/Intermediate_Solutions")
    prob.setParam(GRB.Param.SolFiles, Model_and_Solution_Folder+"/Intermediate_Solutions/")

    
    prob.setParam(GRB.Param.Cutoff, known_Lower_Bound) # This tells Gurobi: “Ignore any solution worse than this.”
    
    
    #https://support.gurobi.com/hc/en-us/community/posts/8791754564369-How-does-MIPFocus-really-work-
    print("\nDefault MIP Focus: ", prob.Params.MIPFocus) # Is 0
    prob.Params.MIPFocus = 2
    print("\nSet MIP Focus: ", prob.Params.MIPFocus)
    # https://www.gurobi.com/documentation/current/refman/mipfocus.html


    # Gurobi needs to be made to use the starting solution being provided
    #https://docs.gurobi.com/projects/optimizer/en/12.0/reference/attributes/variable.html#attrstart
    # prob.setParam("StartNodeLimit", 0) # Check feasibility only (✅ best for strong starts)
    prob.setParam("StartNodeLimit", 25) # Explore 10 nodes around start — for slightly improving near-feasible starts
    

    


    prob.update()
    # Save model structure
    prob.write(f"N_{current_N}/Model_for_N_{current_N}.lp")
    print(f"LP Model saved inside {Model_and_Solution_Folder} for N={current_N}")  
    # Optimize the model with the custom callback function
    prob.optimize(callback_fun)
    
    
    
    
    
    
    
    # Storing the solution in Excel within the correct folder
    wb = openpyxl.Workbook()
    
    # Get workbook active sheet from the active attribute
    sheet_for_J = wb.active
    # Renaming active sheet
    sheet_for_J.title = "J"
    
    row_number_for_sheet_J = 1
    cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 1)
    cell.value = "Value of j"
    cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 2)
    cell.value = "Value of Variable x"
    cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 3)
    cell.value = "Value of Variable f"
    
    # Variable Solutions
    for j in range(1, upper_Bound+1):
        row_number_for_sheet_J += 1
        cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 1)
        cell.value = j
        cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 2)
        cell.value = x[j].X
        # cell.value = model.getVarByName(f"x[{j}]").x
        # cell.value = model.getVarByName(f"x[{j}]").X
        cell = sheet_for_J.cell(row = row_number_for_sheet_J, column = 3)
        cell.value = f[j].X
        # cell.value = model.getVarByName(f'f[{j}]').X
    
    
    
    
    
    # Creating a new Sheet
    sheet_for_JS = wb.create_sheet('JS')
    
    row_number_for_sheet_JS = 1
    cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 1)
    cell.value = "Value of j"
    cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 2)
    cell.value = "Value of s"
    cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 3)
    cell.value = "Value of Variable z"
    
    # Variable Solutions
    for j in range(1, S_comparison_with_X_UpperBound+1):
        for s in permS:
            row_number_for_sheet_JS += 1
            cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 1)
            cell.value = j
            cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 2)
            cell.value = str(s)
            cell = sheet_for_JS.cell(row = row_number_for_sheet_JS, column = 3)
            cell.value = z[s,j].X
            # cell.value = model.getVarByName(f'z[{s},{j}]').X
    
    
    
    
    
    
    
        
    # Creating another new Sheet
    sheet_for_JSI = wb.create_sheet('JSI')
    
    row_number_for_sheet_JSI = 1
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 1)
    cell.value = "Value of j"
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 2)
    cell.value = "Value of s"
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 3)
    cell.value = "Value of i"
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 4)
    cell.value = "Value of Variable y"
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 5)
    cell.value = "Value of Variable g"
    cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 6)
    cell.value = "Value of Variable h"
    
    # Variable Solutions
    for j in range(1, S_comparison_with_X_UpperBound+1):
        for s in permS:
            for i in elements:
                row_number_for_sheet_JSI += 1
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 1)
                cell.value = j
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 2)
                cell.value = str(s)
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 3)
                cell.value = i
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 4)
                cell.value = y[s,j,i].X
                # cell.value = model.getVarByName(f'y[{s},{j},{i}]').X
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 5)
                cell.value = g[s,j,i].X
                # cell.value = model.getVarByName(f'g[{s},{j},{i}]').X
                cell = sheet_for_JSI.cell(row = row_number_for_sheet_JSI, column = 6)
                cell.value = h[s,j,i].X
                # cell.value = model.getVarByName(f'h[{s},{j},{i}]').X
    
    
    
    wb.save(f'{Model_and_Solution_Folder}/Final Solutions.xlsx')

In [ ]:
Model_and_Solution_Folder = f"N_{current_N}"
os.mkdir(Model_and_Solution_Folder)
print(f"Folder created at {Model_and_Solution_Folder}")

from time import time
start_time = time()
model_development()
end_time = time()
print("*****\n Time Elapsed (mins): ", round((end_time-start_time)/60))

Folder created at N_5
Set parameter Username
Set parameter LicenseID to value 2625358
Academic license - for non-commercial use only - expires 2026-02-20
Set parameter SolFiles to value "N_5/Intermediate_Solutions/"
Set parameter Cutoff to value 152

Default MIP Focus:  0
Set parameter MIPFocus to value 2

Set MIP Focus:  2
Set parameter StartNodeLimit to value 25
LP Model saved inside N_5 for N=5
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i3-10110U CPU @ 2.10GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
Cutoff  152
MIPFocus  2
StartNodeLimit  25

Optimize a model with 661987 rows, 286386 columns and 1842405 nonzeros
Model fingerprint: 0x3bd2b186
Variable types: 89400 continuous, 196986 integer (196833 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+08]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 

### This version of the LP has:
1) Improved Upper Bound (i.e. the no. of spaces for the variables x), and
2) Use an existing starting solution (for current_N = 4, 5, ...)